# Notebook 02 — Mathematical Experiments
## Convergence Analysis of Gradient Descent for Logistic Regression

**Purpose:** Run the four controlled experiments from the blueprint and
analyse convergence behaviour.

**Experiments:**
- A: Learning-rate sensitivity
- B: Feature scaling effect
- C: L2 regularisation effect
- D: Independent implementation check (sklearn comparison)


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_preparation import load_raw_data, prepare_data
from src.logistic_regression import (
    binary_cross_entropy, gradient, gradient_norm,
    hessian, hessian_eigenvalues, condition_number,
)
from src.gradient_descent import batch_gradient_descent, run_experiment_grid
from src.evaluation import optimisation_summary
from src.visualization import (
    plot_loss_curves_comparison, plot_grad_norms_comparison,
    plot_eigenvalue_spectrum, plot_scaling_comparison,
    plot_regularisation_comparison,
)

SEED = 42
np.random.seed(SEED)
%matplotlib inline


---
## Experiment A: Learning-Rate Sensitivity

**Setup:** Scaled features, no regularisation, zero initialisation.
Vary η ∈ {0.001, 0.01, 0.1, 0.5, 1.0}.
All other settings fixed.


In [ ]:
# Load and prepare data (scaled)
df = load_raw_data("../data/raw")
data = prepare_data(df, test_size=0.20, random_state=SEED, scale=True)

X_train = data["X_train"]
y_train = data["y_train"]
X_test  = data["X_test"]
y_test  = data["y_test"]

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")


In [ ]:
# Run GD for several learning rates
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]

lr_results = run_experiment_grid(
    X_train, y_train,
    learning_rates=learning_rates,
    lam=0.0,
    max_iter=10000,
    tol=1e-6,
    verbose=True,
)


In [ ]:
# Summary table
rows = []
for lr, res in lr_results.items():
    s = optimisation_summary(res)
    rows.append({"η": lr, **s})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))


In [ ]:
# Loss curves comparison
fig = plot_loss_curves_comparison(
    lr_results,
    title="Experiment A: Loss vs Iteration by Learning Rate",
    save_name="exp_a_loss_curves",
    max_iters_plot=3000,
)
plt.show()


In [ ]:
# Gradient norm comparison
fig = plot_grad_norms_comparison(
    lr_results,
    title="Experiment A: Gradient Norm vs Iteration",
    save_name="exp_a_grad_norms",
    max_iters_plot=3000,
)
plt.show()


### Experiment A — Discussion

*Interpret the results here:*
- Which learning rates converged? How many iterations?
- Did any diverge? What happened to the loss?
- Relate to the theoretical condition η < 2/L.


---
## Experiment B: Feature Scaling

**Setup:** Compare convergence with and without standardisation.
Fixed learning rate η = 0.1, no regularisation.


In [ ]:
# Unscaled data
data_unscaled = prepare_data(df, test_size=0.20, random_state=SEED, scale=False)

# Run GD — unscaled
print("=== UNSCALED ===")
res_unscaled = batch_gradient_descent(
    data_unscaled["X_train"], data_unscaled["y_train"],
    learning_rate=0.01,   # smaller η needed when unscaled
    max_iter=10000, tol=1e-6, verbose=True, print_every=1000,
)

# Run GD — scaled (already have data from Experiment A)
print("\n=== SCALED ===")
res_scaled = batch_gradient_descent(
    X_train, y_train,
    learning_rate=0.1,
    max_iter=10000, tol=1e-6, verbose=True, print_every=1000,
)


In [ ]:
# Side-by-side plots
fig = plot_scaling_comparison(res_unscaled, res_scaled, save_name="exp_b_scaling")
plt.show()


In [ ]:
# Hessian condition numbers
print(f"Condition number (unscaled): {res_unscaled.condition_number:.2f}")
print(f"Condition number (scaled):   {res_scaled.condition_number:.2f}")

# Eigenvalue spectra
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if res_unscaled.hessian_eigenvalues is not None:
    axes[0].bar(range(len(res_unscaled.hessian_eigenvalues)),
                np.sort(res_unscaled.hessian_eigenvalues)[::-1])
    axes[0].set_title(f"Unscaled (κ = {res_unscaled.condition_number:.1f})")
    axes[0].set_ylabel("Eigenvalue")

if res_scaled.hessian_eigenvalues is not None:
    axes[1].bar(range(len(res_scaled.hessian_eigenvalues)),
                np.sort(res_scaled.hessian_eigenvalues)[::-1])
    axes[1].set_title(f"Scaled (κ = {res_scaled.condition_number:.1f})")
    axes[1].set_ylabel("Eigenvalue")

fig.suptitle("Hessian Eigenvalue Spectra", fontsize=13)
fig.tight_layout()
fig.savefig("../results/figures/exp_b_eigenvalues.png", dpi=150, bbox_inches="tight")
plt.show()


### Experiment B — Discussion

*Interpret:*
- How does scaling affect the condition number κ(H)?
- Which converged faster, and by how much?
- Relate the eigenvalue spread to the convergence rate.


---
## Experiment C: L2 Regularisation

**Setup:** Scaled features, η = 0.1.  Compare λ = 0 vs λ = 1.0.


In [ ]:
LAM = 1.0

print("=== Unregularised (λ=0) ===")
res_unreg = batch_gradient_descent(
    X_train, y_train,
    learning_rate=0.1, lam=0.0,
    max_iter=10000, tol=1e-6, verbose=True, print_every=1000,
)

print("\n=== Regularised (λ=1.0) ===")
res_reg = batch_gradient_descent(
    X_train, y_train,
    learning_rate=0.1, lam=LAM,
    max_iter=10000, tol=1e-6, verbose=True, print_every=1000,
)


In [ ]:
fig = plot_regularisation_comparison(res_unreg, res_reg, lam=LAM,
                                    save_name="exp_c_regularisation")
plt.show()

print(f"Condition number (λ=0):   {res_unreg.condition_number:.2f}")
print(f"Condition number (λ={LAM}): {res_reg.condition_number:.2f}")


### Experiment C — Discussion

*Interpret:*
- How does L2 regularisation change the condition number?
- Does it provide strong convexity? Why?
- How does it affect convergence speed?
- **Note:** the regularised and unregularised objectives are different
  functions — compare convergence behaviour, not raw loss values.


---
## Experiment D: Independent Implementation Check

Compare our GD solution against scikit-learn LogisticRegression.


In [ ]:
from src.evaluation import compare_with_sklearn

# Use the best-converged result (scaled, η=0.1, λ=0)
best_res = lr_results[0.1]

comparison = compare_with_sklearn(
    X_train, y_train, X_test, y_test,
    w_gd=best_res.w, b_gd=best_res.b,
    lam=0.0,
)

print("=== Implementation Comparison ===")
print(f"sklearn test accuracy:      {comparison[sklearn_accuracy]:.4f}")
print(f"Max |w_gd - w_sk|:          {comparison[max_w_diff]:.6f}")
print(f"Mean |w_gd - w_sk|:         {comparison[mean_w_diff]:.6f}")
print(f"Max |P(y=1)_gd - P(y=1)_sk|: {comparison[max_prob_diff]:.6f}")
print(f"Mean |P(y=1)_gd - P(y=1)_sk|:{comparison[mean_prob_diff]:.6f}")


In [ ]:
# Coefficient comparison scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(comparison["sklearn_w"], best_res.w, alpha=0.6, s=30)
lims = [min(comparison["sklearn_w"].min(), best_res.w.min()) - 0.1,
        max(comparison["sklearn_w"].max(), best_res.w.max()) + 0.1]
ax.plot(lims, lims, "r--", linewidth=1, label="y = x")
ax.set_xlabel("sklearn coefficients")
ax.set_ylabel("Our GD coefficients")
ax.set_title("Coefficient Comparison: GD vs sklearn")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig("../results/figures/exp_d_coeff_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Experiment D — Discussion

*Interpret:*
- Are the coefficients close?  If not, why? (different solver, tolerance, objective convention)
- Are the predicted probabilities close?
- This is a sanity check, not a correctness proof.


---
## Summary of Mathematical Experiments

| Experiment | Key Finding |
|------------|-------------|
| A: Learning rate | *fill in* |
| B: Feature scaling | *fill in* |
| C: Regularisation | *fill in* |
| D: sklearn check | *fill in* |
